### 05. 결과 시각화·최종 표 (P4)

우선대응 지역과 정책카드를 확인하고, 보고서에 실을 지도·표를 만든다.
그림은 파이프라인 노드가 이미 저장한 것을 그대로 쓰고, 여기서는 표와 추가 그림만 만든다.

In [1]:
import json
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.pipeline.graph import Graph
from src.pipeline.report import node_metrics, node_table
from src.visualization import style

GRAPH = Graph.load()
LAYERS = ROOT / "data/processed/layers"
TABLES = ROOT / "reports/tables"
FIGURES = ROOT / "reports/figures"
style.apply()
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 40)

### P4 노드 상태

In [2]:
node_table(*GRAPH.select(phase='P4'))

,이름,게이트,상태,검증,승인,실패 시,node_id
0,데이터 접근신청 현황,H00,pass,code / human,R1,h00_access_requests,h00_access_requests
1,강수량 수집 확인,H00,pass,code / human,-,h00_collect_rainfall,h00_collect_rainfall
2,하천수위 수집 확인,H00,pass,code / human,-,h00_collect_river,h00_collect_river
3,펌프장·하천 목록 수집 확인,H00,pass,code / human,-,h00_collect_small_tables,h00_collect_small_tables
4,SGIS 인구 통계·경계 수집 확인,H00,pass,code / human,-,h00_collect_sgis,h00_collect_sgis
5,토지피복·DEM 수집 확인,H00,pass,code / human,-,h00_collect_geo,h00_collect_geo
6,강수량 원본 검증,H01,pass,code / code / human,R1,h00_collect_rainfall,h01_contract_rainfall
7,하천수위 원본 검증,H01,pass,code / code / human,R1,h00_collect_river,h01_contract_river
8,펌프장·하천 목록 원본 검증,H01,pass,code / code / human,R1,h00_collect_small_tables,h01_contract_small_tables
9,SGIS 인구 통계·경계 원본 검증,H01,pass,code / code / human,R1,h00_collect_sgis,h01_contract_sgis


## 1. 우선대응 지역

CDRI 상위 격자에서 300m 안 중복을 제거해 선정했다. `ranking_mode` 가 `tier` 이므로
**표의 `rank` 는 표시 순서일 뿐 정밀 순위가 아니다.** 근거는 `grade_final`(R1~R5)과 `in_robust_core` 다.

In [3]:
m8 = node_metrics("h08_top20_policy")
print("ranking_mode :", m8["ranking_mode"])
print("후보 격자     :", f"{m8['n_candidates']:,}", "→ 선정", m8["n_selected"],
      "| 강건 공통집합 포함", m8["n_in_robust_core"])
print("NMS 반경     :", m8["nms_radius_m"], "m")
print()
print("영향 인구:", m8["people_covered"])

ranking_mode : tier
후보 격자     : 10,202 → 선정 20 | 강건 공통집합 포함 6
NMS 반경     : 300.0 m

영향 인구: {'pop_total': 7931, 'elderly_estimate': 3049, 'shelter_over_2km': 0}


In [4]:
top = pd.read_csv(TABLES / "top20.csv", encoding="utf-8-sig")
top[["rank", "district", "neighborhood", "cdri", "grade_code", "grade_raw", "grade_final",
     "in_robust_core", "primary_cause", "pop_total", "elderly_estimate", "shelter_dist_m"]]

,rank,district,neighborhood,cdri,grade_code,grade_raw,grade_final,in_robust_core,primary_cause,pop_total,elderly_estimate,shelter_dist_m
0,1,의창구,북면,1.0000,R5,5,5,0,D,273,124.0,1759.0
1,2,진해구,덕산동,0.9818,R5,5,5,1,E,426,130.0,1072.0
2,3,마산회원구,내서읍,0.9678,R5,5,5,1,V,373,373.0,315.0
3,4,마산합포구,합포동,0.9580,R5,5,5,1,E,382,135.0,439.0
4,5,마산회원구,회원1동,0.9538,R5,5,5,1,H,376,173.0,398.0
5,6,진해구,풍호동,0.9508,R5,5,5,1,H,390,113.0,718.0
6,7,진해구,덕산동,0.9406,R5,5,5,0,E,269,103.0,714.0
7,8,성산구,가음정동,0.9353,R5,5,5,1,E,828,284.0,489.0
8,9,진해구,자은동,0.9351,R5,5,5,0,E,319,120.0,167.0
9,10,진해구,웅천동,0.9243,R5,5,5,0,D,345,142.0,963.0


**주 원인별 조치 — 정책카드의 핵심.** 같은 위험도라도 원인이 다르면 담당과 조치가 다르다.

In [5]:
top.groupby(["primary_cause", "owner_department", "recommended_action"]).agg(
    격자수=("rank", "size"), 인구=("pop_total", "sum"), 고령=("elderly_estimate", "sum")
).reset_index()

,primary_cause,owner_department,recommended_action,격자수,인구,고령
0,D,창원시 재난안전대책본부,반경 2km 내 임시 대피장소 추가 지정과 방재기관 접근로 사전 점검,3,921,385.0
1,E,창원시 재난안전대책본부 (구청 안전건설과 협조),호우경보 시 지하공간·지하차도 출입통제와 도로 우회 안내를 우선 ...,10,4284,1461.0
2,H,창원시 하수도사업소 (해당 하수처리구역 관할),"우기 전 빗물받이·측구 준설, 호우주의보 시 이동식 펌프·차수판 ...",5,2105,698.0
3,V,행정복지센터·복지정책과,고령가구 사전 연락 명단을 갱신하고 호우주의보 시 1차 연락 대상...,2,621,505.0


**정책카드 한 장 전체.** 보고서에는 이 형식으로 20장을 부록에 싣는다.

In [6]:
card = top.iloc[0]
for k, v in card.items():
    print(f"{k:<32} {v}")

rank                             1
grid_id                          라마982027
district                         의창구
neighborhood                     북면
cdri                             1.0
grade_raw                        5
grade_final                      5
grade_code                       R5
grade_name                       R5 최우선 대응
in_robust_core                   0
hazard_contribution              0.178
exposure_contribution            0.3317
vulnerability_contribution       0.2834
capacity_deficit_contribution    0.2069
primary_cause                    D
confidence_grade                 B
trigger                          기상청 호우주의보 (3시간 60mm 또는 12시간 110mm 예상) → 기상청 호우경보 (3시간 90mm 또는 12시간 180mm 예상)
action_timing                    우기 전 상시 + 호우주의보 발효 시
recommended_action               반경 2km 내 임시 대피장소 추가 지정과 방재기관 접근로 사전 점검
owner_department                 창원시 재난안전대책본부
required_resource                기존 시설 협약 (학교·경로당 등)
cost_band                        기존 인력
kpi                       

## 2. 구별 분포와 편중 점검

선정 지역이 특정 구에 몰렸는지 본다. 몰렸다면 그 자체가 결과일 수도 있고
자료 편향일 수도 있으므로, 9/14 침수흔적도로 반드시 대조해야 한다.

In [7]:
by_gu = top.groupby("district").agg(
    선정=("rank", "size"), 인구=("pop_total", "sum"), 고령=("elderly_estimate", "sum")
).sort_values("선정", ascending=False)
cdri = gpd.read_file(LAYERS / "cdri.gpkg", layer="cdri")
GU = {"38111": "의창구", "38112": "성산구", "38113": "마산합포구", "38114": "마산회원구", "38115": "진해구"}
by_gu["순위대상 격자"] = cdri["gu_code"].astype(str).map(GU).value_counts()
by_gu["선정 비율"] = (by_gu["선정"] / by_gu["순위대상 격자"] * 1000).round(2)
by_gu

,선정,인구,고령,순위대상 격자,선정 비율
district,,,,,
진해구,9,2975,1016.0,1653,5.44
마산합포구,4,1799,629.0,2833,1.41
마산회원구,3,1480,770.0,1403,2.14
의창구,3,849,350.0,3039,0.99
성산구,1,828,284.0,1274,0.78


In [8]:
fig, ax = style.new_axes("구별 우선대응 선정 수", "괄호 안은 순위대상 격자 1,000개당 선정 수")
labels = [f"{i}\n({v:.1f})" for i, v in zip(by_gu.index, by_gu["선정 비율"])]
ax.bar(labels, by_gu["선정"])
ax.set_ylabel("선정 격자 수")
style.save(fig, FIGURES / "top20_by_district.png")

findfont: Failed to find font weight bold, now using 400.


## 3. 위험 등급 R1~R5

등급은 **1~5 오름차순이며 5가 가장 위험**하다 (`docs/CDRI_GRADE_SYSTEM.md`, 결정 003).
환경부 하수관로 상태등급과 방향을 맞춰 하수도사업소가 쓰는 언어와 일치시켰다.

본안은 조건부다. 침수흔적 양성 격자가 100개 이상이면 발생률 캘리브레이션이지만
지금은 0개라 **Jenks 자연구분이 본안**이고 나머지는 병기한다. 코드가 판정한다.

In [9]:
from src.data.grades import GRADE_CODES, GRADE_COLORS

m7 = node_metrics("h07_cdri")["grade_system"]
breaks = m7["jenks_breaks"]
fig, ax = style.new_axes("CDRI 분포와 등급 경계", "세로선 = Jenks 본안 경계 (R1~R5)")
ax.hist(cdri["cdri"], bins=60, color="0.75")
for b in breaks[:-1]:
    ax.axvline(b, color="0.3", linewidth=0.9, linestyle="--")
for g in (1, 2, 3, 4, 5):
    lo = 0 if g == 1 else breaks[g - 2]
    ax.axvspan(lo, breaks[g - 1], color=GRADE_COLORS[g], alpha=0.25)
    ax.text((lo + breaks[g - 1]) / 2, ax.get_ylim()[1] * 0.92, GRADE_CODES[g],
            ha="center", fontsize=9)
ax.set_xlabel("CDRI")
ax.set_ylabel("격자 수")
style.save(fig, FIGURES / "cdri_distribution.png")

In [10]:
# 방식별 등급 분포와 일치도. Balica 는 R1 이 1칸뿐이라 본안으로 쓸 수 없다.
summary = pd.DataFrame({
    "Jenks(본안)": {k: v["n"] for k, v in m7["raw"].items()},
    "규칙 적용 후": {k: v["n"] for k, v in m7["final"].items()},
    "고정 백분위": {k: v["n"] for k, v in m7["percentile_for_comparison"].items()},
    "Balica": {k: v["n"] for k, v in m7["balica_for_comparison"].items()},
    "설계 목표": {k: round(v["target_share"] * len(cdri)) for k, v in m7["raw"].items()},
}).reindex(["R5", "R4", "R3", "R2", "R1"])
print("가중 kappa:", m7["weighted_kappa"])
summary

가중 kappa: {'jenks_vs_balica': 0.7199, 'jenks_vs_percentile': 0.8104, 'balica_vs_percentile': 0.4875, 'note': '2차 가중 kappa. Landis & Koch(1977) 기준 0.61~0.80 substantial'}


,Jenks(본안),규칙 적용 후,고정 백분위,Balica,설계 목표
R5,624,675,205,226,204
R4,1650,1661,816,1746,816
R3,2551,2524,2040,5831,2040
R2,3165,3137,3060,2398,3061
R1,2212,2205,4081,1,4081


## 4. 보고서에 실을 그림

파이프라인 노드가 저장한 그림이다. 노트북에서 다시 그리지 않는다.

| 파일 | 내용 | 만든 노드 |
|---|---|---|
| `stations_map.png` | 관측지점 31곳 | `h03_stations` |
| `grid_features_check.png` | 격자 변수 6종 | 확인용 |
| `layer1_map.png` | 노출·민감도·등급·L1 | `h06_layer1_flood` |
| `layer3_map.png` | E·V·대응역량 부족도 | `h06_layer3_vuln` |
| `cdri_map.png` | CDRI·주 원인 | `h07_cdri` |
| `eda_data_check/` | 데이터 확인 EDA 4장 | `h02_eda_data_check` |

In [11]:
for p in sorted(FIGURES.rglob("*.png")):
    print(f"{p.relative_to(ROOT)}  ({p.stat().st_size // 1024} KB)")

reports/figures/cdri_distribution.png  (50 KB)
reports/figures/cdri_map.png  (176 KB)
reports/figures/eda_data_check/rainfall_annual_totals.png  (51 KB)
reports/figures/eda_data_check/rainfall_station_coverage.png  (84 KB)
reports/figures/eda_data_check/river_level_validity.png  (43 KB)
reports/figures/eda_data_check/sgis_population_map.png  (56 KB)
reports/figures/grid_features_check.png  (929 KB)
reports/figures/layer1_map.png  (636 KB)
reports/figures/layer3_map.png  (321 KB)
reports/figures/stations_map.png  (190 KB)
reports/figures/top20_by_district.png  (40 KB)


## 5. 아직 못 넣은 것

- **외적 검증** — 실제 침수와 대조한 적이 없다. 9/14 침수흔적도 수령 후 ROC-AUC·사후검증
- **행정동 이름** — 표의 `neighborhood` 가 "행정동명 미확보"다. `data/external/adm_dong_names.csv` 필요
- **Layer 2** — 관로 비공개로 CDRI 에서 제외 (`docs/decisions/001-layer2-design.md`)